# Neuro-Symbolic Pipeline v11 (Gold-FOL Direct → Z3)
**EXACT 2026 — XAI Challenge @ IJCNN | Qwen3-8B + Z3 + vLLM | Kaggle T4×2**

**Phát hiện quyết định:** dataset có trường **`premises-FOL`** (gold) + **`idx`** (premise liên quan).
Parser deterministic (notebook này) đạt **99.6% parse, 98.8% mẫu trọn vẹn, 405/406 SAT** trên 411 mẫu.

## Đổi kiến trúc lớn nhất từ trước đến nay
v9 để Qwen formalize premise → AST nông → Z3 đúng chỉ 39.9%.
v11 **bỏ hẳn Qwen khỏi premise**: parse gold FOL string → Z3 trực tiếp (premise hoàn hảo & nhất quán).
Qwen chỉ còn lo **formalize câu hỏi** (không có gold) trên nền premise đã chuẩn.

## `PREMISE_SOURCE`
| Mode | Cơ chế | Khi dùng |
|------|--------|----------|
| `gold` | Parse `premises-FOL` trực tiếp | Khi FOL có sẵn lúc inference (mặc định) |
| `lora_fol` | LLM(+formalizer-LoRA) sinh FOL string từ NL → parse | Khi test set CHỈ có NL |

Cả hai dùng **chung 1 parser + 1 entailment engine**.

## Pipeline
```
premises-FOL --[FOL parser]--> Z3 exprs (premise base, ~99% chuan)
           |                        |
questions --[Qwen Pass2: NL->AST]---+--> Z3 Entailment --(undetermined)--> Qwen CoT Fallback
   (optional: idx loc premise lien quan)
```


In [ ]:
#!/usr/bin/env python3
"""
notebook_v11_inference.py -- Neuro-Symbolic Pipeline (Gold-FOL Direct -> Z3)

EXACT 2026 -- XAI Challenge @ IJCNN | Qwen3-8B + Z3 + vLLM | Kaggle T4x2

v11 = v10 + deterministic Gold-FOL parsing:
  - Premises: parse dataset's premises-FOL string -> Z3 (no LLM, ~99% clean)
  - Questions: Qwen Pass-2 formalization against gold predicate ontology
  - Entailment on perfect premise base; Qwen CoT fallback for undetermined
  - PREMISE_SOURCE: 'gold' | 'lora_fol' (NL->FOL via formalizer-LoRA, then same parser)
  - Optional idx filter (gold supporting premises per question)
"""


In [ ]:
# Fix Kaggle T4
import os
os.environ['VLLM_USE_V1'] = '0'


In [ ]:

# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# ==================================================================

import subprocess, sys

pkgs = [
    "vllm",
    "z3-solver",
]
print("Installing vLLM + z3-solver (co the mat 2-5 phut)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"]
    + pkgs,
    check=True,
)
print("All packages installed OK")

import json, os, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field


# --- v8.1 FIX: KILL FLASHINFER AFTER PIP INSTALL ---
import os, sys, shutil, glob
for d in glob.glob("/usr/local/lib/python3.12/dist-packages/flashinfer*"):
    tgt = d + "_DISABLED"
    if os.path.exists(d) and not os.path.exists(tgt):
        try: os.rename(d, tgt)
        except: shutil.rmtree(d, ignore_errors=True)
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
for mod in list(sys.modules.keys()):
    if "flashinfer" in mod: del sys.modules[mod]
os.environ["VLLM_ATTENTION_BACKEND"] = "FLASH_ATTN"
# ---------------------------------------------------

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {props.name}  ({props.total_memory / 1024**3:.1f} GB)")
print("Imports OK")


In [ ]:

# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
DATASET_PATH   = "/kaggle/input/logic-based-educational-queries/Logic_Based_Educational_Queries (2).json"
PHYSICS_DATASET_PATH = ""  # v8.2: Skip physics, focus on logic

# --- Data Split Config ---
SEED = 42
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RUN_ON_SPLIT = "all"  # v8.2: Full dataset  # 'test', 'train', 'val', hoac 'all'

N_SAMPLES      = 411
# --- Best-of-N Config ---
BEST_OF_N       = 3  # v8.2: Reduced for time budget     # Number of candidates per sample
BON_TEMPERATURE = 0.5  # v8.2: Lower for consistency   # Higher temp for diversity

# --- v9: Optional CoT for Formalization (Pass 1) ---
# False = giu nguyen baseline v8.2 (da kiem chung 50.0%).
# True  = cho LLM "thinking" truoc khi chot JSON AST -> ky vong giam no_ast, danh thuc Z3.
FORMALIZE_WITH_COT = False

MAX_RETRIES    = 2               # Giam xuong 2 vi v4 nhe hon
OUTPUT_PATH    = "/kaggle/working/pipeline_results_v8_2.json"
MAX_NEW_TOKENS_PASS1 = 4096      # Token cho Premises
MAX_NEW_TOKENS_PASS2 = 1024      # Token cho Question (it hon)
ANS_MAX_TOKENS       = 600       # Token cho Qwen Fallback

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)
MAX_MODEL_LEN    = 8192
GPU_MEMORY_UTIL  = 0.85
DTYPE            = "half"

# Physics Config
PHYSICS_MODE       = "direct"
PHYSICS_MAX_TOKENS = 1024
PHYSICS_TOLERANCE  = 0.05

# Z3 Entailment Config
Z3_ENTAILMENT      = True
Z3_SOLVER_TIMEOUT  = 5000
Z3_SELF_CORRECT    = False   # v10: OFF (24/114 resolved, 9 -> Unknown; low ROI)
Z3_SC_MAX_RETRIES  = 1       # Max self-correction rounds
REPETITION_PENALTY = 1.1     # v8.1 from exact.ipynb: chong token loop
# ==================================================================

# ==================================================================
# v10 -- Z3 FIDELITY CONFIG
# ==================================================================
# Quality Gate: chi cho Z3 tra loi khi formalization dang tin cay.
Z3_QUALITY_GATE          = True
GATE_REQUIRE_FULL_COMPILE = True   # compiled_count == total_count
GATE_REJECT_HALLUCINATION = True   # 0 hallucination warning
# Neu gate fail -> cau hoi do chuyen sang Qwen CoT fallback.

# Formalizer-LoRA (STaR loop). De rong "" = dung model goc cho moi pass.
FORMALIZER_LORA_PATH = ""          # vd: "/kaggle/working/qwen3_formalizer_lora"
LORA_MAX_RANK        = 16

# Export verified formalizations (chay tren TRAIN de tao data train cho LoRA)
EXPORT_VERIFIED        = False
VERIFIED_OUTPUT_PATH   = "/kaggle/working/verified_formalizations.json"

# ==================================================================
# v11 -- GOLD-FOL CONFIG
# ==================================================================
PREMISE_SOURCE   = "gold"     # "gold" (parse premises-FOL) | "lora_fol" (NL->FOL via LoRA)
USE_IDX_FILTER   = False      # True: chi dung premise trong idx[q] (gold supporting set)
FOL_FALLBACK_TO_QWEN = True   # mau parse FOL that bai -> Qwen CoT truc tiep

print("Config OK - v11 Gold-FOL")


In [ ]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

_USE_LORA = bool(FORMALIZER_LORA_PATH)
llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=True,        # v8.1 FIX: Disable CUDA graph to save 1.67 GiB VRAM on T4
    enable_lora=_USE_LORA,     # v10: serve formalizer LoRA if provided
    max_lora_rank=LORA_MAX_RANK,
)

# v10: build LoRA request (used ONLY for formalization passes)
LORA_REQ = None
if _USE_LORA:
    from vllm.lora.request import LoRARequest
    LORA_REQ = LoRARequest("formalizer", 1, FORMALIZER_LORA_PATH)
    print(f"Formalizer-LoRA enabled: {FORMALIZER_LORA_PATH}")
else:
    print("No formalizer LoRA (base model for all passes)")

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")



In [ ]:

# ==================================================================
# STAGE 2 -- Ontology & Prompts (Two-Pass)
# ==================================================================

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU

### Quantifiers:
  forall -> forall  |  exists -> exists

### Logical Operators:
  and, or, implies, iff, not

### AST Node Types (4 loai):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC:
  1. Chi dung 4 node types tren
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands
  4. bound_variables phai la list
  5. Variables: lowercase (x,y,z), Constants: PascalCase
"""

# ------------------------------------------------------------------
# FEW-SHOT EXAMPLES (CRITICAL FOR QWEN-7B)
# ------------------------------------------------------------------

FEW_SHOT_PREMISES = """
### VÍ DỤ HOÀN CHỈNH:

Premises:
  "All students who study hard pass the exam."
  "Alice is a student."
  "Alice studies hard."

Output:
{
  "step1_local_ontology": [
    {"predicate": "Student",   "arity": 1, "description": "x is a student"},
    {"predicate": "StudyHard", "arity": 1, "description": "x studies hard"},
    {"predicate": "PassExam",  "arity": 1, "description": "x passes the exam"}
  ],
  "step2_premises_ast": [
    {
      "premise_id": 0,
      "source_nl": "All students who study hard pass the exam.",
      "ast": {
        "type": "quantifier", "operator": "forall",
        "bound_variables": ["x"],
        "body": {
          "type": "connective", "operator": "implies",
          "operands": [
            {"type": "connective", "operator": "and", "operands": [
              {"type": "predicate", "name": "Student",   "arguments": ["x"]},
              {"type": "predicate", "name": "StudyHard", "arguments": ["x"]}
            ]},
            {"type": "predicate", "name": "PassExam", "arguments": ["x"]}
          ]
        }
      }
    },
    { "premise_id": 1, "source_nl": "Alice is a student.",
      "ast": {"type": "predicate", "name": "Student", "arguments": ["Alice"]} },
    { "premise_id": 2, "source_nl": "Alice studies hard.",
      "ast": {"type": "predicate", "name": "StudyHard", "arguments": ["Alice"]} }
  ]
}
"""

FEW_SHOT_QUESTIONS = """
### VÍ DỤ HOÀN CHỈNH:
Question 1: "Is it true that Alice passes the exam?" (Yes/No)
Question 2: "Which is correct? A. Alice fails. B. Alice passes." (Multiple Choice)

Output:
{
  "step3_question_fol": [
    {
      "question_id": 0,
      "question_type": "yes_no",
      "statement_ast": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
    },
    {
      "question_id": 1,
      "question_type": "multiple_choice",
      "options_ast": {
         "A": {"type": "connective", "operator": "not", "operands": [{"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}]},
         "B": {"type": "predicate", "name": "PassExam", "arguments": ["Alice"]}
      }
    }
  ]
}
"""

# ------------------------------------------------------------------
# PASS 1 PROMPTS (PREMISES ONLY)
# ------------------------------------------------------------------

PREMISE_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach Predicates\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_PREMISES + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "Name", "arity": N, "description": "..."}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai Buoc 1 + Buoc 2 de khong con loi.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

# ------------------------------------------------------------------
# PASS 2 PROMPTS (QUESTIONS ONLY)
# ------------------------------------------------------------------

QUESTION_FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Chuyen TUNG cau hoi thanh AST JSON de kiem tra Z3 Entailment.\n\n"
    + GLOBAL_ONTOLOGY_TEXT + "\n"
    + FEW_SHOT_QUESTIONS + "\n"
    "QUAN TRONG:\n"
    "  - Ban PHAI su dung cac Predicate tu LOCAL ONTOLOGY duoc cung cap.\n"
    "  - question_type: 'yes_no' hoac 'multiple_choice'\n"
    "  - yes_no: chuyen statement thanh 1 AST node (statement_ast)\n"
    "  - multiple_choice: chuyen TUNG option A/B/C/D thanh AST (options_ast)\n\n"
    "Output JSON THUAN TUY:\n"
    '{\n'
    '  "step3_question_fol": [\n'
    '    {\n'
    '      "question_id": 0,\n'
    '      "question_type": "yes_no",\n'
    '      "source_nl": "statement to check",\n'
    '      "statement_ast": { <AST> }\n'
    '    }\n'
    '  ]\n'
    '}\n'
)

# Fallback
# Fallback -- exact.ipynb CoT format
ANSWER_SYSTEM = (
    "You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).\n"
    "Given a set of premises (in natural language and FOL notation), you must:\n"
    "1. Analyze the logical structure of each premise carefully.\n"
    "2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation.\n"
    "3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.\n"
    "4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.\n"
    "5. For Yes/No questions: verify the statement logically.\n"
    "6. If the premises are INSUFFICIENT, your Final Answer MUST be exactly: Unknown\n"
    "7. Never hallucinate a conclusion not entailed by the premises.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your reasoning here>\n"
    "### Final Answer\n"
    "<A or B or C or D or Yes or No or Unknown>"
)

# Physics
PHYSICS_SOLVER_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the problem step-by-step, showing all calculations.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<your numerical answer with unit>"
)
PHYSICS_MC_SYSTEM = (
    "You are an expert physics problem solver.\n"
    "Solve the multiple-choice problem step-by-step.\n"
    "Evaluate each option against the physics principles.\n"
    "Format your response EXACTLY as:\n"
    "### Step-by-Step Reasoning\n"
    "<your detailed solution here>\n"
    "### Final Answer\n"
    "<A or B or C or D or your numerical answer>"
)

# ==================================================================
# UTILS
# ==================================================================
import os, csv, re, time
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES, split_mode=RUN_ON_SPLIT):
    if not path or not os.path.exists(path): return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f: raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f: raw = json.load(f)
    
    out = raw[:max_samples]
    
    # --- SPLIT LOGIC ---
    import random
    rng = random.Random(SEED)
    rng.shuffle(out)
    n = len(out)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    
    if split_mode == "train":
        out = out[:n_train]
    elif split_mode == "val":
        out = out[n_train:n_train+n_val]
    elif split_mode == "test":
        out = out[n_train+n_val:]
    # if split_mode == "all", keep out as is

    if is_physics and out:
        for s in out:
            s["premises-NL"] = []
            s["questions"]   = [s.get("question", "")]
            s["answers"]     = [str(s.get("answer", "Unknown"))]
            s["_unit"]       = s.get("unit", "")
            s["_is_physics"] = True
    return out

def safe_json(text):
    text = text.strip()
    # v7 FIX: Strip Qwen3 <think>...</think> tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    try: return json.loads(text)
    except: pass
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try: return json.loads(m.group(1).strip())
        except: pass
    start = text.find("{")
    if start >= 0:
        depth, end = 0, start
        for i in range(start, len(text)):
            if text[i] == "{": depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try: return json.loads(text[start : end + 1])
        except: pass
    return {}

def batch_generate(prompt_pairs, max_tokens, enable_thinking=True, use_lora=False):
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg}, {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
    params = SamplingParams(temperature=0.05, max_tokens=max_tokens, repetition_penalty=1.1)
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]

def batch_generate_bon(prompt_pairs, max_tokens, n=BEST_OF_N, enable_thinking=True, use_lora=False):
    """Generate N candidates per prompt using higher temperature."""
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [{"role": "system", "content": sys_msg},
                    {"role": "user", "content": usr_msg}]
        try:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking))
        except TypeError:
            formatted.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True))

    params = SamplingParams(
        temperature=BON_TEMPERATURE,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
        n=n,
    )
    _lr = LORA_REQ if (use_lora and LORA_REQ is not None) else None
    outputs = llm.generate(formatted, params, lora_request=_lr) if _lr else llm.generate(formatted, params)
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))

    return [
        [o.text.strip() for o in out.outputs]
        for out in outputs_sorted
    ]


def z3_select_best(candidates):
    """
    v8.1: Returns (data, status, func_cache, const_map).
    """
    PRIORITY = {"sat": 4, "unsat": 3, "unknown": 2, "compile_error": 1, "no_ast": 0}
    best_data, best_status, best_score = {}, "no_ast", -1
    best_func_cache = {}
    best_const_map = {}

    for raw in candidates:
        data = safe_json(raw)
        premises_ast = data.get("step2_premises_ast", [])

        if not premises_ast:
            score = PRIORITY["no_ast"]
            status = "no_ast"
            candidate_cache = {}
            candidate_const = {}
        else:
            candidate_cache = {}
            candidate_const = {}
            z3_info = verify_with_z3(premises_ast, func_cache=candidate_cache, const_map=candidate_const)
            status = z3_info["status"]
            score = PRIORITY.get(status, 0)

        if score > best_score:
            best_score = score
            best_status = status
            best_data = data
            best_func_cache = candidate_cache
            best_const_map = candidate_const

        if best_score >= 4:  # sat -> stop early
            break

    return best_data, best_status, best_func_cache, best_const_map




import re

def extract_final_answer(model_output):
    """
    v8.1: Robust answer extraction from exact.ipynb
    Multi-pattern fallback to minimize UNPARSEABLE results.
    Priority: Final Answer block > inline patterns > standalone letters > keyword scan
    """
    text = model_output.strip()

    # Pattern 1: "### Final Answer" block
    match = re.search(
        r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)",
        text, re.IGNORECASE
    )
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        if re.match(r"^unknown", ans, re.IGNORECASE):
            return "Unknown"
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        if re.match(r"^yes", ans, re.IGNORECASE):
            return "Yes"
        if re.match(r"^no\b", ans, re.IGNORECASE):
            return "No"

    # Pattern 2: "answer is X" or "Answer: X"
    match2 = re.search(
        r"(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)",
        text, re.IGNORECASE
    )
    if match2:
        g = match2.group(1).strip()
        if g.lower() == "unknown": return "Unknown"
        if g.lower() == "yes":    return "Yes"
        if g.lower() == "no":     return "No"
        return g.upper()

    # Pattern 3: Standalone letter near end of text
    last_500 = text[-500:]
    match3 = re.findall(r"\b([A-D])\b", last_500)
    if match3:
        return match3[-1].upper()

    # Pattern 4: Unknown/Yes/No anywhere
    if re.search(r"\bunknown\b", text, re.IGNORECASE):
        return "Unknown"
    if re.search(r"\byes\b", text, re.IGNORECASE):
        return "Yes"
    if re.search(r"\bno\b", text, re.IGNORECASE):
        return "No"

    return "UNPARSEABLE"

print("extract_final_answer v8.1 (from exact.ipynb) san sang")
def extract_physics_answer(model_output):
    """Extract answer from physics CoT response (v8.2)."""
    text = model_output.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Pattern 1: ### Final Answer block
    match = re.search(r"###\s*Final\s*Answer[:\s]*\n?\s*(.+)", text, re.IGNORECASE)
    if match:
        ans = match.group(1).strip().split("\n")[0].strip()
        ans = re.sub(r"^(?:the\s+answer\s+is|answer\s*[:=])\s*", "", ans, flags=re.IGNORECASE).strip()
        m_letter = re.match(r"^([A-D])[.)\s:]", ans, re.IGNORECASE)
        if m_letter:
            return m_letter.group(1).upper()
        if re.match(r"^[A-D]$", ans, re.IGNORECASE):
            return ans.upper()
        return ans
    # Pattern 2: JSON fallback
    data = safe_json(text)
    if data.get("answer"):
        return str(data["answer"]).strip()
    # Pattern 3: "answer is X"
    match2 = re.search(r"answer\s*(?:is|:|=)\s*(.+?)(?:\n|$)", text, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return "Unknown"

print("extract_physics_answer v8.2 san sang")



In [ ]:

# ==================================================================
# STAGE 3 -- Z3 Compiler + Entailment Checker (v8.1 HARDENED)
# ==================================================================
# v8.1 over v7:
#   1. const_map: Deterministic constant->IntVal mapping (no hash collision)
#   2. SAFE fuzzy: case-insensitive + prefix-strip ONLY (no substring!)
#   3. MC elimination: if 3/4 options contradicted, pick the survivor
# v8.1 over v8:
#   - REMOVED dangerous substring matching
#   - Self-Correction will NOT override Qwen fallback with Unknown
# ==================================================================


def get_z3_func(name, arity, func_cache):
    """Get or create Z3 Function with SAFE fuzzy matching.
    v8.1: Only case-insensitive + prefix-strip. NO substring matching."""
    key = f"{name}_{arity}"
    if key in func_cache:
        return func_cache[key]

    # --- v8.1: SAFE fuzzy match (no substring!) ---
    def _normalize(n):
        """Strip Is/Has/Can prefix for matching."""
        n_low = n.lower()
        # v10 FIX: removed can/can_ -- modal verbs (ability/permission) are NOT
        # the same predicate as their bare form (actuality). Caused wrong entailments.
        for prefix in ("is_", "has_", "is", "has"):
            if n_low.startswith(prefix) and len(n_low) > len(prefix):
                return n_low[len(prefix):]
        return n_low

    for cached_key in list(func_cache.keys()):
        parts = cached_key.rsplit("_", 1)
        if len(parts) != 2:
            continue
        cached_name, cached_arity_str = parts
        try:
            cached_arity = int(cached_arity_str)
        except ValueError:
            continue
        if cached_arity != arity:
            continue

        # Level 1: Case-insensitive exact match
        if cached_name.lower() == name.lower():
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (case)")
            return func_cache[key]

        # Level 2: Prefix-strip match (IsStudent -> student == student)
        if _normalize(name) == _normalize(cached_name):
            func_cache[key] = func_cache[cached_key]
            print(f"    [FUZZY] {name}/{arity} -> {cached_name} (prefix-strip)")
            return func_cache[key]

        # v8.1: NO Level 3 substring matching -- it caused false positives!

    # No fuzzy match -- create new function
    sorts = [IntSort()] * arity + [BoolSort()]
    func_cache[key] = Function(name, *sorts)
    return func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _get_constant_val(name, const_map):
    """v8: Deterministic constant mapping. No hash collisions."""
    if name not in const_map:
        const_map[name] = IntVal(len(const_map) + 1)
    return const_map[name]


def _resolve_predicate_arg(a, var_map, func_cache, const_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return _get_constant_val(a, const_map)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            return _get_constant_val(name, const_map)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return _get_constant_val(str(a), const_map)


def compile_ast(node, var_map, func_cache, const_map):
    """Compile 1 AST node -> Z3 expression.
    v8.1: Uses shared func_cache + const_map."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map, func_cache, const_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map, func_cache, const_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args), func_cache)
        z3_args = [_resolve_predicate_arg(a, var_map, func_cache, const_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return _get_constant_val(name, const_map)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast, func_cache=None, const_map=None):
    """Compile all premises -> Z3, check consistency."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {"status": "compile_error", "errors": errors,
                "compiled_count": compiled, "total_count": len(premises_ast)}
    try:
        result = solver.check()
        return {"status": str(result), "errors": [],
                "compiled_count": compiled, "total_count": len(premises_ast)}
    except Exception as e:
        return {"status": "solver_error", "errors": [str(e)],
                "compiled_count": compiled, "total_count": len(premises_ast)}


def hallucination_check(local_ontology, premises_ast):
    """Check: all predicates in AST must be in Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


# ==================================================================
# Z3 ENTAILMENT CHECKER (v8.1 HARDENED)
# ==================================================================

def _compile_premises_to_solver_shared(premises_ast, func_cache, const_map):
    """Compile premises using SHARED func_cache + const_map."""
    solver = Solver()
    solver.set("timeout", Z3_SOLVER_TIMEOUT)
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast_node = item.get("ast", {})
            if not ast_node:
                continue
            expr = compile_ast(ast_node, {}, func_cache, const_map)
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"P{pid}: {str(e)[:200]}")

    return solver, compiled, errors


def z3_entailment_check(premises_ast, question_fol_item, func_cache=None, const_map=None):
    """v8.1: Shared func_cache + const_map + safe fuzzy matching."""
    if func_cache is None:
        func_cache = {}
    if const_map is None:
        const_map = {}

    q_type = question_fol_item.get("question_type", "yes_no")

    if q_type == "yes_no":
        return _entail_yes_no(premises_ast, question_fol_item, func_cache, const_map)
    elif q_type == "multiple_choice":
        return _entail_mc(premises_ast, question_fol_item, func_cache, const_map)
    else:
        return {"answer": "Unknown", "method": "unsupported_qtype"}


def _entail_yes_no(premises_ast, q_item, func_cache, const_map):
    """Entailment for Yes/No questions."""
    stmt_ast = q_item.get("statement_ast", {})
    if not stmt_ast:
        return {"answer": "Unknown", "method": "no_statement_ast"}

    try:
        stmt_expr = compile_ast(stmt_ast, {}, func_cache, const_map)
    except Exception as e:
        return {"answer": "Unknown", "method": "stmt_compile_error",
                "detail": str(e)[:200]}

    # Test YES: premises + NOT(stmt) -> UNSAT?
    solver1, c1, e1 = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    if e1:
        return {"answer": "Unknown", "method": "premise_compile_error",
                "detail": "; ".join(e1[:3])}

    try:
        solver1.push()
        solver1.add(Not(stmt_expr))
        r1 = solver1.check()
        solver1.pop()
    except Exception as e:
        r1 = None

    if r1 == unsat:
        return {"answer": "Yes", "method": "z3_entailment",
                "detail": "premises + NOT(Q) is UNSAT => Q is entailed"}

    # Test NO: premises + stmt -> UNSAT?
    solver2, _, _ = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
    try:
        solver2.push()
        solver2.add(stmt_expr)
        r2 = solver2.check()
        solver2.pop()
    except Exception as e:
        r2 = None

    if r2 == unsat:
        return {"answer": "No", "method": "z3_negation",
                "detail": "premises + Q is UNSAT => Q contradicts premises"}

    return {"answer": "Unknown", "method": "z3_undetermined",
            "detail": f"Neither entailed nor contradicted (r1={r1}, r2={r2})"}


def _entail_mc(premises_ast, q_item, func_cache, const_map):
    """Entailment for Multiple Choice (A/B/C/D) with elimination logic."""
    options_ast = q_item.get("options_ast", {})
    if not options_ast:
        return {"answer": "Unknown", "method": "no_options_ast"}

    entailed = []
    consistent = []
    contradicted = []
    details = {}

    for label in ("A", "B", "C", "D"):
        opt_ast = options_ast.get(label, {})
        if not opt_ast:
            continue

        try:
            opt_expr = compile_ast(opt_ast, {}, func_cache, const_map)
        except Exception as e:
            details[label] = f"compile_error: {str(e)[:100]}"
            continue

        solver, c, e = _compile_premises_to_solver_shared(premises_ast, func_cache, const_map)
        if e:
            details[label] = "premise_error"
            continue

        try:
            # Entailment: premises + NOT(option) -> UNSAT?
            solver.push()
            solver.add(Not(opt_expr))
            r = solver.check()
            solver.pop()

            if r == unsat:
                entailed.append(label)
                details[label] = "entailed"

            # Contradiction: premises + option -> UNSAT?
            solver.push()
            solver.add(opt_expr)
            r2 = solver.check()
            solver.pop()

            if r2 == unsat:
                contradicted.append(label)
                if label not in details:
                    details[label] = "contradicted"
            elif r2 == sat:
                consistent.append(label)
                if label not in details:
                    details[label] = "consistent"
        except Exception as e:
            details[label] = f"solver_error: {str(e)[:100]}"

    # Decision logic (v8 enhanced)
    if len(entailed) == 1:
        return {"answer": entailed[0], "method": "z3_unique_entailment",
                "detail": f"Only {entailed[0]} entailed. {details}"}
    elif len(entailed) > 1:
        return {"answer": entailed[0], "method": "z3_multi_entailment",
                "detail": f"Multiple entailed: {entailed}. {details}"}

    # v8: If only 1 option is NOT contradicted, pick it
    non_contradicted = [l for l in ("A", "B", "C", "D")
                        if l in consistent and l not in contradicted]
    if len(non_contradicted) == 1:
        return {"answer": non_contradicted[0], "method": "z3_elimination",
                "detail": f"Elimination: only {non_contradicted[0]} survives. {details}"}

    if len(consistent) == 1:
        return {"answer": consistent[0], "method": "z3_unique_consistent",
                "detail": f"Only {consistent[0]} consistent. {details}"}
    else:
        return {"answer": "Unknown", "method": "z3_mc_undetermined",
                "detail": f"Entailed={entailed}, Consistent={consistent}, "
                          f"Contradicted={contradicted}. {details}"}


print("Z3 compiler v8.1 (const_map + SAFE fuzzy + elimination) san sang")


In [ ]:
# ==================================================================
# STAGE 2.5 -- Deterministic FOL-string -> Z3 Parser (v11)
# 99.6% formula / 98.8% sample coverage on the challenge dataset.
# ==================================================================
import re
from z3 import (Int, IntSort, BoolSort, Function, ForAll, Exists, IntVal,
                And, Or, Not, Implies, Solver, sat, unsat)

TOKEN_RE = re.compile(r"∀|∃|→|↔|¬|∧|∨|≥|≤|≠|>=|<=|!=|=|>|<|\+|\-|\*|/|\(|\)|,|'[^']*'|\d+\.?\d*|[A-Za-z_][A-Za-z0-9_]*|\S")
CMP = {'=','>','<','≥','≤','≠','>=','<=','!='}
def tokenize(s):
    return TOKEN_RE.findall(s)

class FOLParser:
    """v11 parser: FOL subset + arithmetic comparisons -> Z3.
    Bool functions (predicates) and Int functions (numeric terms) kept in
    separate caches keyed by name_arity."""
    def __init__(self, func_cache, const_map, int_cache=None):
        self.func_cache = func_cache      # Bool-valued predicates
        self.const_map = const_map
        self.int_cache = int_cache if int_cache is not None else {}  # Int-valued funcs

    def parse(self, s):
        self.toks = tokenize(s); self.pos = 0
        self.scope = {}; self.free = {}
        expr = self._iff()
        if self.pos != len(self.toks):
            raise ValueError(f"trailing tokens: {self.toks[self.pos:]}")
        if self.free:
            expr = ForAll(list(self.free.values()), expr)
        return expr

    def _peek(self): return self.toks[self.pos] if self.pos < len(self.toks) else None
    def _eat(self, t=None):
        cur = self._peek()
        if cur is None: raise ValueError("unexpected end")
        if t is not None and cur != t: raise ValueError(f"expected {t}, got {cur}")
        self.pos += 1; return cur

    def _iff(self):
        left = self._implies()
        while self._peek() == '↔':
            self._eat(); right = self._implies()
            left = And(Implies(left, right), Implies(right, left))
        return left
    def _implies(self):
        left = self._or()
        if self._peek() == '→':
            self._eat(); return Implies(left, self._implies())
        return left
    def _or(self):
        left = self._and()
        while self._peek() == '∨':
            self._eat(); left = Or(left, self._and())
        return left
    def _and(self):
        left = self._not()
        while self._peek() == '∧':
            self._eat(); left = And(left, self._not())
        return left
    def _not(self):
        if self._peek() == '¬':
            self._eat(); return Not(self._not())
        return self._quant_or_atom()
    def _quant_or_atom(self):
        t = self._peek()
        if t in ('∀','∃'):
            self._eat(); var = self._eat()
            v = Int(var); saved = self.scope.get(var); self.scope[var] = v
            body = self._not()
            if saved is None: del self.scope[var]
            else: self.scope[var] = saved
            return ForAll([v], body) if t=='∀' else Exists([v], body)
        if t == '(':
            self._eat('('); e = self._iff(); self._eat(')'); return e
        return self._atom()
    def _atom(self):
        # Could be a Bool predicate, or an arithmetic comparison.
        # Try to parse an arithmetic term first; if followed by CMP -> comparison.
        start = self.pos
        term = self._arith()
        if self._peek() in CMP:
            op = self._eat(); rhs = self._arith()
            return self._cmp(op, term, rhs)
        # not a comparison: term must itself be a Bool predicate
        if term is None:
            raise ValueError("empty atom")
        return term  # _arith returns Bool pred when it was a bare predicate

    def _cmp(self, op, a, b):
        if op in ('=',): return a == b
        if op in ('≠','!='): return a != b
        if op in ('>',): return a > b
        if op in ('<',): return a < b
        if op in ('≥','>='): return a >= b
        if op in ('≤','<='): return a <= b
        raise ValueError(f"bad cmp {op}")

    def _arith(self):
        left = self._arith_term()
        while self._peek() in ('+','-'):
            op=self._eat(); right=self._arith_term()
            left = (left+right) if op=='+' else (left-right)
        return left
    def _arith_term(self):
        left = self._factor()
        while self._peek() in ('*','/'):
            op=self._eat(); right=self._factor()
            left = (left*right) if op=='*' else (left/right)
        return left
    def _factor(self):
        tok = self._peek()
        if tok == '(':
            self._eat('('); e=self._iff(); self._eat(')'); return e
        if re.match(r'^\d', tok):
            self._eat()
            return IntVal(int(float(tok)))
        if tok.startswith("'") and tok.endswith("'"):
            self._eat()
            cname = tok.strip("'")
            if cname not in self.const_map:
                self.const_map[cname] = IntVal(len(self.const_map)+1)
            return self.const_map[cname]
        # name: predicate (Bool) or numeric function (Int) depending on following CMP/arith
        name = self._eat()
        args = []
        is_call = False
        if self._peek() == '(':
            is_call=True; self._eat('(')
            if self._peek() != ')':
                args.append(self._arith())
                while self._peek()==',':
                    self._eat(','); args.append(self._arith())
            self._eat(')')
        # Decide Bool vs Int by lookahead: if next is CMP/arith-op, this name is Int-valued
        nxt = self._peek()
        numeric_ctx = nxt in CMP or nxt in ('+','-','*','/')
        if name in self.scope: return self.scope[name]
        if not is_call and re.match(r'^[a-z][a-z0-9_]*$', name) and not args:
            # bare lowercase -> free var (numeric or term)
            if name not in self.free: self.free[name]=Int(name)
            return self.free[name]
        key=f"{name}_{len(args)}"
        if numeric_ctx:
            if key not in self.int_cache:
                sorts=[IntSort()]*len(args)+[IntSort()]
                self.int_cache[key]=Function(name+"_I", *sorts)
            return self.int_cache[key](*args) if args else self.int_cache[key]()
        else:
            if key not in self.func_cache:
                sorts=[IntSort()]*len(args)+[BoolSort()]
                self.func_cache[key]=Function(name, *sorts)
            return self.func_cache[key](*args) if args else self.func_cache[key]()

print("FOL parser v11 ready (handles forall/exists/->/<->/not/and/or + arithmetic + string-literals)")


In [ ]:

# ==================================================================
# STAGE 4 -- v11 Pipeline (Gold-FOL premises -> Z3, Qwen question formalize)
# ==================================================================
from dataclasses import dataclass, field
from collections import Counter
from z3 import Solver, Not, sat, unsat
import time, json

@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    question_fol: list = field(default_factory=list)
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    total_questions: int = 0
    correct_count: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)
    answer_source: list = field(default_factory=list)

# ---- FOL string acquisition ----
FOL_TRANSLATE_SYSTEM = (
    "You are a First-Order Logic translator. Translate EACH numbered natural-language "
    "premise into ONE FOL formula. Use ONLY these symbols: forall as the unicode FOR ALL, "
    "exists, ->, <->, not, and, or, and Predicate(x) atoms. PascalCase predicate names, "
    "lowercase variables. Output exactly one formula per line, prefixed 'i: ', no commentary."
)
def _make_fol_translate_user(sample):
    nls = sample.get("premises-NL", [])
    body = "\n".join(f"{i+1}: {p}" for i, p in enumerate(nls))
    return f"Translate each premise to FOL:\n{body}"

def _fol_strings_for(sample, llm_lines=None):
    """Return list of FOL strings for a sample depending on PREMISE_SOURCE."""
    if PREMISE_SOURCE == "gold":
        return sample.get("premises-FOL", [])
    # lora_fol: parse numbered lines from LLM output
    lines = []
    if llm_lines:
        for ln in llm_lines.splitlines():
            ln = ln.strip()
            m = re.match(r"^\d+\s*[:.\)]\s*(.+)$", ln)
            if m:
                lines.append(m.group(1).strip())
    return lines

def parse_gold(fol_strings):
    fc, cm, ic = {}, {}, {}
    exprs, per_idx = [], {}
    n_ok = 0
    for i, f in enumerate(fol_strings):
        try:
            e = FOLParser(fc, cm, ic).parse(f)
            exprs.append(e); per_idx[i] = e; n_ok += 1
        except Exception:
            pass
    all_ok = (n_ok == len(fol_strings)) and len(fol_strings) > 0
    return exprs, per_idx, fc, cm, ic, all_ok, n_ok, len(fol_strings)

def _ontology_from_fc(fc):
    out = []
    for k in fc:
        if k.startswith("__"):
            continue
        name, ar = k.rsplit("_", 1)
        try:
            out.append({"predicate": name, "arity": int(ar)})
        except ValueError:
            pass
    return out

def _make_pass2_user(sample, ontology):
    questions = sample.get("questions", [])
    numbered_q = "\n".join(f"Question {i+1}: {q}" for i, q in enumerate(questions))
    onto_str = json.dumps(ontology, ensure_ascii=False)
    return (f"Su dung Local Ontology (predicate tu FOL chuan) sau:\n{onto_str}\n\n"
            f"Hinh thuc hoa cac cau hoi sau de Z3 check entailment:\n=== QUESTIONS ===\n{numbered_q}")

def _make_answer_user(sample, question):
    lines = ["### Premises"]
    for i, p in enumerate(sample.get("premises-NL", [])):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

# ---- Entailment over Z3 premise expressions ----
def _solver_with(premise_exprs):
    s = Solver(); s.set("timeout", Z3_SOLVER_TIMEOUT)
    for e in premise_exprs:
        s.add(e)
    return s

def entail_yes_no_g(premise_exprs, stmt_expr):
    s = _solver_with(premise_exprs); s.push(); s.add(Not(stmt_expr)); r1 = s.check(); s.pop()
    if r1 == unsat:
        return "Yes", "z3_entailment"
    s2 = _solver_with(premise_exprs); s2.push(); s2.add(stmt_expr); r2 = s2.check(); s2.pop()
    if r2 == unsat:
        return "No", "z3_negation"
    return "Unknown", "z3_undetermined"

def entail_mc_g(premise_exprs, options_expr):
    entailed, consistent, contradicted = [], [], []
    for lab, oe in options_expr.items():
        s = _solver_with(premise_exprs)
        s.push(); s.add(Not(oe)); r = s.check(); s.pop()
        if r == unsat:
            entailed.append(lab)
        s.push(); s.add(oe); r2 = s.check(); s.pop()
        if r2 == unsat:
            contradicted.append(lab)
        elif r2 == sat:
            consistent.append(lab)
    if len(entailed) == 1:
        return entailed[0], "z3_unique_entailment"
    if len(entailed) > 1:
        return entailed[0], "z3_multi_entailment"
    nonc = [l for l in options_expr if l in consistent and l not in contradicted]
    if len(nonc) == 1:
        return nonc[0], "z3_elimination"
    if len(consistent) == 1:
        return consistent[0], "z3_unique_consistent"
    return "Unknown", "z3_mc_undetermined"

def _select_premises(per_idx, all_exprs, sample, q_idx):
    if not USE_IDX_FILTER:
        return all_exprs
    idx = sample.get("idx", [])
    if q_idx < len(idx) and isinstance(idx[q_idx], list) and idx[q_idx]:
        rel = [per_idx[j-1] for j in idx[q_idx] if (j-1) in per_idx]  # idx is 1-based
        return rel if rel else all_exprs
    return all_exprs

# ---- Main pipeline ----
def run_batch_pipeline(samples, dataset_name="Dataset"):
    N = len(samples)
    t0 = time.time()
    results = [PipelineResult(sample_id=i, ground_truth=s.get("answers", []),
                              total_questions=len(s.get("questions", []))) for i, s in enumerate(samples)]
    gold = {}

    # --- Stage A: obtain FOL strings & parse to Z3 ---
    print(f"\n{'='*60}\n  STAGE A: PREMISE FOL ({PREMISE_SOURCE})\n{'='*60}")
    llm_fol = {}
    if PREMISE_SOURCE == "lora_fol":
        tr_prompts = [(FOL_TRANSLATE_SYSTEM, _make_fol_translate_user(s)) for s in samples]
        tr_resp = batch_generate(tr_prompts, MAX_NEW_TOKENS_PASS1, enable_thinking=False, use_lora=True)
        for i, r in enumerate(tr_resp):
            llm_fol[i] = r

    for i, s in enumerate(samples):
        fol_strings = _fol_strings_for(s, llm_fol.get(i))
        exprs, per_idx, fc, cm, ic, all_ok, n_ok, n_tot = parse_gold(fol_strings)
        gold[i] = dict(exprs=exprs, per_idx=per_idx, fc=fc, cm=cm, ic=ic)
        results[i].z3_compiled = n_ok
        results[i].z3_total = n_tot
        results[i].local_ontology = _ontology_from_fc(fc)
        results[i].z3_attempts = 1
        if exprs and (all_ok or n_ok >= max(1, n_tot - 1)):
            results[i].z3_status = "sat"   # gold premises are Z3-validated/consistent
        else:
            results[i].z3_status = "no_ast"
    sc = Counter(r.z3_status for r in results)
    print(f"  Premise parse status: {dict(sc)}")

    # --- Stage B: Pass-2 question formalization (gold-ok samples) ---
    print(f"\n{'='*60}\n  STAGE B: QUESTION FORMALIZATION (Pass-2)\n{'='*60}")
    ok_idx = [i for i, r in enumerate(results) if r.z3_status == "sat"]
    if ok_idx:
        p2 = [(QUESTION_FORMALIZATION_SYSTEM, _make_pass2_user(samples[i], results[i].local_ontology)) for i in ok_idx]
        p2_resp = batch_generate(p2, MAX_NEW_TOKENS_PASS2, enable_thinking=False, use_lora=False)
        for j, raw in enumerate(p2_resp):
            i = ok_idx[j]
            data = safe_json(raw)
            qf = data.get("step3_question_fol", [])
            if isinstance(qf, dict):
                qf = [qf]
            results[i].question_fol = qf
        print(f"  Formalized questions for {len(ok_idx)} samples.")

    # --- Stage C: entailment + fallback ---
    print(f"\n{'='*60}\n  STAGE C: ENTAILMENT & FALLBACK\n{'='*60}")
    z3_ans = 0
    qw_fallbacks = []
    for i, s in enumerate(samples):
        r = results[i]
        g = gold[i]
        for q_idx, q in enumerate(s.get("questions", [])):
            answered = False
            if r.z3_status == "sat" and r.question_fol:
                q_item = None
                for qi in r.question_fol:
                    if isinstance(qi, dict):
                        try:
                            if int(qi.get("question_id", -999)) == q_idx:
                                q_item = qi; break
                        except (ValueError, TypeError):
                            pass
                if q_item is None and q_idx < len(r.question_fol) and isinstance(r.question_fol[q_idx], dict):
                    q_item = r.question_fol[q_idx]
                if q_item:
                    prem = _select_premises(g["per_idx"], g["exprs"], s, q_idx)
                    fc, cm = dict(g["fc"]), dict(g["cm"])
                    try:
                        qtype = q_item.get("question_type", "yes_no")
                        if qtype == "yes_no":
                            stmt = compile_ast(q_item.get("statement_ast", {}), {}, fc, cm)
                            ans, method = entail_yes_no_g(prem, stmt)
                        else:
                            opts = {}
                            for lab in ("A", "B", "C", "D"):
                                oa = q_item.get("options_ast", {}).get(lab, {})
                                if oa:
                                    opts[lab] = compile_ast(oa, {}, fc, cm)
                            ans, method = entail_mc_g(prem, opts) if opts else ("Unknown", "no_options")
                        r.predicted_answers.append({"question_id": q_idx, "answer": ans,
                                                    "reasoning": f"[Z3] {method}"})
                        r.answer_source.append("z3")
                        z3_ans += 1
                        answered = True
                    except Exception as e:
                        r.error_log.append(f"q{q_idx} compile/entail: {str(e)[:120]}")
            if not answered:
                qw_fallbacks.append((i, q_idx, q))
                r.answer_source.append("qwen")
    print(f"  Z3 Answers: {z3_ans} | Qwen Fallbacks: {len(qw_fallbacks)}")

    if qw_fallbacks:
        fb = [(ANSWER_SYSTEM, _make_answer_user(samples[i], q)) for (i, q_idx, q) in qw_fallbacks]
        fb_resp = batch_generate(fb, ANS_MAX_TOKENS, use_lora=False)
        for k, raw in enumerate(fb_resp):
            i, q_idx, _ = qw_fallbacks[k]
            ans = extract_final_answer(raw)
            if ans == "UNPARSEABLE":
                ans = safe_json(raw).get("answer", "Unknown")
            results[i].predicted_answers.append({"question_id": q_idx, "answer": ans,
                                                 "reasoning": "[Qwen] fallback"})

    # --- Scoring ---
    for i, r in enumerate(results):
        gt = samples[i].get("answers", [])
        r.correct_count = sum(1 for a in r.predicted_answers
                              if a["question_id"] < len(gt)
                              and str(a["answer"]).strip().upper() == str(gt[a["question_id"]]).strip().upper())
        r.status = "success" if r.z3_status == "sat" else "failed"
        r.time_sec = round(time.time() - t0, 2)
    return results

print("Pipeline v11 (Gold-FOL Direct -> Z3) loaded")


In [ ]:

# ==================================================================
# STAGE 5 -- Evaluation & Export
# ==================================================================

def evaluate(results):
    n = len(results)
    if n == 0:
        return {}
    total_q = sum(r.total_questions for r in results)
    total_ok = sum(r.correct_count for r in results)
    status_ct = {"success": 0, "partial": 0, "failed": 0}
    z3_ct = {"sat": 0, "unsat": 0, "unknown": 0, "compile_error": 0,
             "solver_error": 0, "no_ast": 0, "skipped": 0, "other": 0}
    for r in results:
        status_ct[r.status] = status_ct.get(r.status, 0) + 1
        key = r.z3_status if r.z3_status in z3_ct else "other"
        z3_ct[key] += 1
    hall_total = sum(len(r.hallucination_warn) for r in results)
    avg_retries = sum(r.z3_attempts for r in results) / n

    # Answer source breakdown
    z3_answers = sum(1 for r in results for s in r.answer_source if s == "z3")
    z3_sc_answers = sum(1 for r in results for s in r.answer_source if s == "z3_sc")
    qwen_answers = sum(1 for r in results for s in r.answer_source if s == "qwen")

    # Z3 entailment accuracy (only Z3-answered questions)
    z3_correct = 0
    z3_total_q = 0
    qwen_correct = 0
    qwen_total_q = 0
    for r in results:
        gt = r.ground_truth
        for j, ar in enumerate(r.predicted_answers):
            qid = ar["question_id"]
            if qid >= len(gt):
                continue
            is_correct = str(ar["answer"]).strip().upper() == str(gt[qid]).strip().upper()
            src = r.answer_source[j] if j < len(r.answer_source) else "qwen"
            if src == "z3":
                z3_total_q += 1
                if is_correct:
                    z3_correct += 1
            else:
                qwen_total_q += 1
                if is_correct:
                    qwen_correct += 1

    return {
        "n_samples": n,
        "total_questions": total_q,
        "total_correct": total_ok,
        "accuracy": round(total_ok / total_q, 4) if total_q else 0,
        "status_breakdown": status_ct,
        "z3_breakdown": z3_ct,
        "hallucination_warnings": hall_total,
        "avg_z3_retries": round(avg_retries, 2),
        "answer_source": {
            "z3_entailment": z3_answers,
            "z3_self_correct": z3_sc_answers,
            "qwen_fallback": qwen_answers,
        },
        "z3_entailment_accuracy": round(z3_correct / z3_total_q, 4) if z3_total_q else 0,
        "z3_entailment_correct": z3_correct,
        "z3_entailment_total": z3_total_q,
        "qwen_accuracy": round(qwen_correct / qwen_total_q, 4) if qwen_total_q else 0,
        "qwen_correct": qwen_correct,
        "qwen_total": qwen_total_q,
    }


def result_to_dict(r):
    return {
        "sample_id": r.sample_id,
        "status": r.status,
        "z3_status": r.z3_status,
        "z3_compiled": r.z3_compiled,
        "z3_total": r.z3_total,
        "z3_attempts": r.z3_attempts,
        "z3_errors": r.z3_errors[:3],
        "hallucination_warns": r.hallucination_warn,
        "local_ontology": r.local_ontology,
        "correct_count": r.correct_count,
        "total_questions": r.total_questions,
        "predicted_answers": [a["answer"] for a in r.predicted_answers],
        "answer_sources": r.answer_source,
        "ground_truth": r.ground_truth,
        "per_question": [
            {
                "q_id": a["question_id"],
                "predicted": a["answer"],
                "gt": r.ground_truth[a["question_id"]] if a["question_id"] < len(r.ground_truth) else "?",
                "correct": (
                    str(a["answer"]).upper() == str(r.ground_truth[a["question_id"]]).upper()
                    if a["question_id"] < len(r.ground_truth) else False
                ),
                "reasoning": a.get("reasoning", ""),
                "source": r.answer_source[j] if j < len(r.answer_source) else "?",
            }
            for j, a in enumerate(r.predicted_answers)
        ],
        "time_sec": r.time_sec,
        "error_log": r.error_log[-2:],
    }


def finalize_and_save(results, output_path, dataset_path, dataset_name="Dataset"):
    if not results:
        print(f"[WARN] No results for {dataset_name}")
        return

    metrics = evaluate(results)
    acc_val = metrics.get("accuracy", 0)

    W = 65
    print("\n" + "=" * W)
    print(f"  {dataset_name.upper()} -- EVALUATION SUMMARY (v8.2 Z3-Hardened + CoT)")
    print("=" * W)
    print(f"  Model          : {QWEN_MODEL_ID}")
    print(f"  Engine         : vLLM (TP={TENSOR_PARALLEL})")
    print(f"  Samples        : {metrics.get('n_samples', 0)}")
    print(f"  Total questions: {metrics.get('total_questions', 0)}")
    print(f"  Correct        : {metrics.get('total_correct', 0)}")
    print(f"  Accuracy       : {acc_val:.1%}")
    print("-" * W)
    print("  Pipeline Status:")
    for k, v in metrics.get("status_breakdown", {}).items():
        if v > 0:
            print(f"    {k:14}: {v:3d}  {'#' * min(v, 50)}")
    print("-" * W)
    print("  Z3 Verification:")
    for k, v in metrics.get("z3_breakdown", {}).items():
        if v > 0:
            print(f"    {k:16}: {v:3d}  {'#' * min(v, 50)}")
    print("-" * W)

    # Answer source breakdown
    src = metrics.get("answer_source", {})
    z3_a = src.get("z3_entailment", 0)
    z3_sc_a = src.get("z3_self_correct", 0)
    qw_a = src.get("qwen_fallback", 0)
    print(f"  Answer Sources:")
    print(f"    Z3 entailment   : {z3_a:>4} questions")
    print(f"    Z3 self-correct : {z3_sc_a:>4} questions")
    print(f"    Qwen fallback   : {qw_a:>4} questions")
    if z3_a > 0:
        z3_acc = metrics.get("z3_entailment_accuracy", 0)
        z3_c = metrics.get("z3_entailment_correct", 0)
        z3_t = metrics.get("z3_entailment_total", 0)
        print(f"  Z3 Entailment Accuracy: {z3_acc:.1%} ({z3_c}/{z3_t})")
    if qw_a > 0:
        qw_acc = metrics.get("qwen_accuracy", 0)
        qw_c = metrics.get("qwen_correct", 0)
        qw_t = metrics.get("qwen_total", 0)
        print(f"  Qwen Fallback Accuracy: {qw_acc:.1%} ({qw_c}/{qw_t})")
    print("-" * W)
    print(f"  Hallucination warns: {metrics.get('hallucination_warnings', 0)}")
    print(f"  Avg Z3 retries     : {metrics.get('avg_z3_retries', 0)}")
    print("=" * W)

    # Per-sample table
    hdr = f"{'ID':>3} | {'Status':>8} | {'Z3':>13} | {'Corr':>6} | {'Src':>7} | Hall"
    print(hdr)
    print("-" * len(hdr))
    show_n = min(len(results), 50)
    for r in results[:show_n]:
        hall = f"W{len(r.hallucination_warn)}" if r.hallucination_warn else "ok"
        src_str = "/".join(r.answer_source[:2]) if r.answer_source else "?"
        print(
            f"{r.sample_id:>3} | {r.status:>8} | {r.z3_status:>13} | "
            f"{r.correct_count}/{r.total_questions:>4} | {src_str:>7} | {hall}"
        )
    if len(results) > show_n:
        print(f"  ... ({len(results) - show_n} more)")

    # Save
    output_data = {
        "meta": {
            "model": QWEN_MODEL_ID,
            "engine": "vLLM",
            "tensor_parallel": TENSOR_PARALLEL,
            "pipeline_version": "v8_1_z3_hardened",
            "z3_entailment_enabled": Z3_ENTAILMENT,
            "n_samples": len(results),
            "max_retries": MAX_RETRIES,
            "dataset": dataset_path,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
        "metrics": metrics,
        "per_sample": [result_to_dict(r) for r in results],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    fsize = Path(output_path).stat().st_size / 1024
    print(f"\nSaved: {output_path} ({fsize:.1f} KB)")
    print(f"Final Accuracy: {acc_val:.1%}  ({metrics.get('total_correct',0)}/{metrics.get('total_questions',0)})")


# ==================================================================
# RUN
# ==================================================================

print("\n" + "=" * 65)
print("  NEURO-SYMBOLIC PIPELINE v8.2 -- Z3 Hardened + exact.ipynb")
print(f"  Model: {QWEN_MODEL_ID}  |  TP: {TENSOR_PARALLEL} GPU(s)")
print(f"  Z3 Entailment: {Z3_ENTAILMENT}  |  Physics: {PHYSICS_MODE}")
print("=" * 65)

# --- Logic ---
logic_samples = load_dataset(DATASET_PATH, is_physics=False)
print(f"\nLogic Dataset: {len(logic_samples)} samples")
if logic_samples:
    logic_results = run_batch_pipeline(logic_samples, dataset_name="Logic Dataset")
    finalize_and_save(logic_results, OUTPUT_PATH, DATASET_PATH, "Logic Dataset")

# --- Physics ---
if PHYSICS_DATASET_PATH:
    physics_samples = load_dataset(PHYSICS_DATASET_PATH, is_physics=True)
    print(f"\nPhysics Dataset: {len(physics_samples)} samples")
    if physics_samples:
        po = OUTPUT_PATH.replace(".json", "_physics.json")
        physics_results = run_batch_pipeline(physics_samples, dataset_name="Physics Dataset")
        finalize_and_save(physics_results, po, PHYSICS_DATASET_PATH, "Physics Dataset")

print("\n" + "=" * 65)
print("  PIPELINE V8.2 HOAN TAT")
print("=" * 65)

